# 4 Datenobjekte abholen


Dieses Script erstellt in 'object' die Unterordner mit dem Identifier, die Files werden ohne weitere Zwischenverarbeitung oder Prüfung hierhin kopiert. 
Das Migrieren sowie Entzippen erfolgt vorerst manuell. 



## Variante A: Datenobjekte liegen auf einem Laufwerk

Gültig für Digitalisate aus der Sosa, bzw. Objekte, die auf lokalen Laufwerken liegen. 
Achtung! Dieses Notebook kann in der Ausführung sehr lange dauern, da die Dateien riesig sind. 
Voraussetzung: mit VPN verbunden / im unilu-Netzwerk, mit Laufwerk G verbunden.

Aufgrund der geringen Menge und der diversen Metadatenquellen werden diese Zipkapseln von Hand vom Laufwerk G auf die Workbench verschoben. Sie werden alle manuell entzippt. 

Die Zipkapseln sind alle nach folgender Struktur benannt:

    DOI/ID _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiele:

    000190118_20150320T000256_master_ver1.zip
    10_7891_e-manuscripta-108732.zip

Die Objekte liegen auf G:\ZHB-Sosa_Digital\digital. Die vollständigen Pfade auf G sind in der Eingabedatei ergänzt und ist in die Infojson unter 'additional' abgelegt. 
Der Dateipfad auf der Workbench wird nach dem DOI benannt. 



In [1]:
import os
import config
import json
from datetime import datetime
import zipfile
import shutil
from pathlib import Path

localdrive = config.user_root
org_id = config.organisation_id
file_name = f'{config.inventory_file}'
objects_path = f'{localdrive}/{config.collection_id}/{config.object_path}'
counter = 0

with open(file_name, encoding="utf-8") as data_file:    
    data = json.load(data_file)
    for value in data:
        counter += 1
        
        # get path to G drive:
        g_path = value["additional"]
        #print(f"Origin path: {g_path}")
        
        # create new object folder name (SIP path). The full path is needed here for copying the files.
        #foldername = value["signature"][(len(org_id)+1):]
        foldername = value["references"][-1]
        sip_path = f"{objects_path}/{foldername}"
         
        print("Destination path:",sip_path)
                
        # prepare object folder: make a directory for each object
        Path(f'{sip_path}').mkdir(parents=True, exist_ok=True)
        
        
        filenames = []
        # Iterate directory, check if current file_path is a file
        try:
            for file_path in os.listdir(g_path):
                print("Origin files:", file_path)
                if os.path.isfile(os.path.join(g_path, file_path)):
                    filenames.append(file_path)
                else:
                    print("---------------- not a file!-----------------------")
        except FileNotFoundError:
            print(f"The directory {g_path} does not exist")
        except PermissionError:
            print(f"Permission denied to access the directory {g_path}")
        except OSError as e:
            print(f"An OS error occurred: {e}")
        
        for file in filenames:   
            sip_file = Path(sip_path+'/'+file)
            if sip_file.exists():
                # path exists
                print("*** Path exists, file already copied")
            else:
                # copy file:
                print("Copying file:", file)
                print("Time started copying:",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))
                shutil.copy(g_path+'/'+file, sip_path+'/'+file) 
            
                print("File copied successfully.")
                
        #debugging:        
        if counter == 1:
            break
        

            
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

Destination path: C:/Users/HeimK/switchdrive/jupyter/dlza/sosa_emanus/objects/10_7891_e-manuscripta-108732
Origin files: 10_7891_e-manuscripta-108732
---------------- not a file!-----------------------
Origin files: 10_7891_e-manuscripta-108732.zip
Copying file: 10_7891_e-manuscripta-108732.zip
Time started copying: 2024-01-05 17:14:01
File copied successfully.
Finished at  2024-01-05 17:14:03


## Variante B: Download aus Zenodo

Die Datenabholung für Zenodo-Repositories funktioniert etwas anders: mittels HTTP download direkt von zenodo.

Test-Daten, Stand Januar 2024:
counter 60-70: enthält ein Datenset (page 7)

### Delay (rate limiting)

Global limit for guest users: 60 requests per minute, 2000 requests per hour
OAI-PMH API harvesting: 120 requests per minute
=> pro loop 1 Sekunde sleep() einbauen. 


In [15]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



  Using cached python_dotenv-1.0.0-py3-none-any.whl (19 kB)


In [4]:
import fitz  # PyMuPDF
import requests
import json
import os
import time
import config
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# get access token
load_dotenv()
ACCESS_TOKEN = os.getenv('access_token')

# get config variables
file_name = config.inventory_file
community = config.collection_id
objects_path = f'{config.user_root}/{config.collection_id}/{config.object_path}'
zenodo_api = config.zenodo_api

download_manually = f'{community}_download_manually.txt'
counter = 0
debug = 20 # adapt for debug mode. For prod: set to 99999

# read inventory file
with open(file_name, encoding="utf-8") as data_file:    
    data = json.load(data_file)
    for value in data:
        
        counter = counter+1
        
        if counter <= debug: 
    
            # prepare object folder: make a directory for each object (SIP)  
            foldername = value["references"][1]
            sip_path = f"{objects_path}/{foldername}"                  
            Path(f'{sip_path}').mkdir(parents=True, exist_ok=True)
            
            # get the necessary identifier
            identifiers = {}
            for item in value["identifiers"]:
                # split identifiers in dict
                [key, value] = item.split(':',1)
                identifiers[key] = value

            # get the file download link from zenodo: 
            zenodo_id = identifiers['zenodo']                   
            zenodo_link = f'{zenodo_api}/{zenodo_id}/files'    
            print(f"\n#{counter}: Get files from:",zenodo_link)
            response = requests.get(zenodo_link, params={'access_token': ACCESS_TOKEN})
            file_object = response.json()

            try: 
                for entry in file_object['entries']:
                    # get file info
                    download_url = entry['links']['content']
                    file_name = entry['key']
                    mimetype = entry['mimetype']

                    print("filename:",file_name, "mimetype:",mimetype)
                    local_file = f'{sip_path}/{file_name}'
                    
                    # download content
                    response = requests.get(download_url, params={'access_token': ACCESS_TOKEN})
                    with open(local_file, mode="wb") as file:
                        file.write(response.content)
                        print(response)
                        print("Downloaded:",local_file)
                        # wait 1 second for every record so as not to overshoot zenodo rate limiting. 
                        time.sleep(1) 

                    # distinguish between pdf files and other files
                    if mimetype == 'application/pdf':
                        # check some stuff on file's integrity, e.g. count pages
                        try:
                            with fitz.open(local_file) as pdf_document:
                                if pdf_document.page_count != 0:
                                    print("PDF page count:",pdf_document.page_count)
                        except Exception as e:
                            print(f"---   Error checking PDF file {file_name}: {e}")
                            # append download url to download_manually.txt
                            with open(download_manually, 'a') as file:
                                file.write(download_url)
                                file.write("\n")
                                print(f"URL {download_url} appended to {download_manually}\n")                                
                        
                    else:
                        print("---   Not a PDF")
                        # append download url to download_manually.txt
                        with open(download_manually, 'a') as file:
                            file.write(download_url)
                            file.write("\n")
                            print(f"URL {download_url} appended to {download_manually}\n")
            except KeyError:
                print("---   KeyError: files not found, download manually:")
                # append download url to download_manually.txt
                with open(download_manually, 'a') as file:
                    file.write(zenodo_link)
                    file.write("\n")
                    print(f"URL {zenodo_link} appended to {download_manually}\n")
        
print("\nFinished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


#1: Get files from: https://zenodo.org/api/records/10466776/files
filename: Cache_WareReinheit.pdf mimetype: application/pdf
<Response [200]>
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10466776/Cache_WareReinheit.pdf
PDF page count: 92

#2: Get files from: https://zenodo.org/api/records/10471407/files
filename: BRAVEpapers_AM.pdf mimetype: application/pdf
<Response [200]>
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10471407/BRAVEpapers_AM.pdf
PDF page count: 12

#3: Get files from: https://zenodo.org/api/records/10471212/files
filename: e077454.full.pdf mimetype: application/pdf
<Response [200]>
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10471212/e077454.full.pdf
PDF page count: 10

#4: Get files from: https://zenodo.org/api/records/10471210/files
filename: e067542.full.pdf mimetype: application/pdf
<Response [200]>
Downloaded: C:/Users/HeimK/switchdrive/jupy